## Python Execution and Memory

### What happens when we press Run?

We write Python as readable text, but the processor cannot execute that text directly. Something has to understand the code, convert it into executable instructions, and manage those instructions while the program is running.

That work is handled by **CPython**, the standard and most commonly used implementation of Python.

#### The complete journey

```text
We write Python source code
             |
             v
Parser reads its structure and creates an AST
             |
             v
Compiler converts the AST into Python bytecode
             |
             v
PVM reads and executes the bytecode instructions
             |
             v
CPython runtime performs the actual operations
```

Let us understand what each stage contributes.

#### 1. Python first understands the structure

The parser reads the source and checks how the code is structured. It converts that structure into an **Abstract Syntax Tree (AST)**. The AST represents ideas such as assignments, function calls, conditions, and expressions in a form that CPython can work with.

If the code breaks Python's grammar, execution stops here with a `SyntaxError`.

#### 2. The compiler creates bytecode

CPython's compiler converts the AST into a **code object** containing bytecode. Bytecode is a compact set of instructions for the Python runtime—for example, load a value, add two values, call a function, or store a result.

Bytecode is not CPU machine code. The processor cannot execute it directly. It is also tied to a Python implementation and version, so bytecode created by one Python version may not work with another.

#### 3. The Python Virtual Machine executes it

The **Python Virtual Machine (PVM)** is the execution engine inside CPython. It reads one bytecode instruction, asks the CPython runtime to perform that operation, and then moves to the next instruction. This continues until the code finishes or an error interrupts it.

The PVM is not a separate program that we install or start. It is one part of the CPython interpreter.

> **Key idea:** The interpreter is the complete system—parser, compiler, PVM, memory manager, and runtime libraries. The PVM is specifically the part that executes bytecode.

### Does Python compile the code every time?

For a normal code cell or top-level script, CPython compiles the code before executing it. Imported modules get an extra optimization: CPython may save their compiled code so it can be reused later.

When we imported `module.py`, CPython created this actual file in our project:

```text
__pycache__/module.cpython-314.pyc
```

Here is what the filename tells us:

- `module` — it came from `module.py`
- `cpython` — CPython created the cache
- `314` — it was created for CPython 3.14
- `.pyc` — it contains cached compiled Python code

The next time the module is imported, CPython checks whether the source file and Python version still match the cache. If the cache is valid, CPython can skip the parsing and compilation stages. The PVM must still execute the cached bytecode—the cache only avoids repeating the earlier work.

Jupyter follows the same general execution process for each code cell, but it does not normally save every cell inside `__pycache__`. Running a top-level `.py` file also normally does not create a `.pyc` cache for that file; imported modules commonly do.

> **Remember:** A `.pyc` file is cached compiled Python code. It is not a standalone executable and it is not CPU machine code.

### Importing modules

**Use module functions:** Import a module, then use dot notation to call the functions it provides.

In [ ]:
import os
os.getcwd()

**Read module attributes:** Dot notation can also access values exposed by an imported module.

In [ ]:
import sys
sys.platform

### Using a local module

**Import your own file:** A file named `module.py` is imported as `module`; the filename without `.py` becomes the module name.

In [ ]:
import module

**Call an imported function:** Access the function with dot notation, then pass the required argument.

In [ ]:
module.greet("Pranav")

**Read an imported variable:** Access the value with dot notation and no parentheses.

In [ ]:
module.topic

**Variable is not callable:** Adding parentheses treats the stored string as a function and raises `TypeError`.

In [ ]:
module.topic()

**Reload a module:** Reloading runs an already imported module again, making recent file changes available without restarting the notebook kernel.

In [ ]:
from importlib import reload
reload(module)

### Mutable and immutable objects

Every value in Python is an object, and variables hold references to those objects.

- **Immutable:** The existing object cannot be changed. Examples: integers, floats, booleans, strings, tuples, frozen sets, and bytes.
- **Mutable:** The existing object can be changed in place. Examples: lists, sets, dictionaries, arrays, and bytearrays.

> Reassigning a variable is allowed in both cases—it changes the reference, not the original immutable object.

**String reassignment:** The string is not modified; `username` is changed to reference a new string object.

```text
Before: username ──► "Pranav"
After:  username ──► "John Doe" (new object)
```

The old object becomes eligible for garbage collection when nothing references it.

In [ ]:
username = "Pranav"
print(username)
username = "John Doe"
print(username)

**Integer reassignment:** `x` and `y` first reference the same object; reassigning `x` does not change `y`.

```text
Before: x ──► 10 ◄── y
After:  x ──► 15     y ──► 10
```

In [ ]:
x = 10
y = x
print(x, y)

x = 15
print(x, y)

In [ ]:
import sys

print(sys.getrefcount(24601))
print(sys.getrefcount("test")) # this string is never initialized before
print(sys.getrefcount(1)) # prints same number as "test"


3
3221225472
3221225472


### Names, objects, and types

In Python, a variable such as `a` is a **name that points to an object in memory**. The object stores the value and knows its type. The name `a` does not contain the object and does not have a permanently fixed type.

**Step 1: `a = 3`**
```text
Name                         Object in memory
+---+      points to         +------------------+
| a | ---------------------> | type: int        |
+---+                        | value: 3         |
                             +------------------+
```

`a` is pointing to an integer object whose value is `3`.

**Step 2: `a = "hello"`**
```text
Name                         Object in memory
+---+      now points to     +------------------+
| a | ---------------------> | type: str        |
+---+                        | value: "hello"  |
                             +------------------+

                             +------------------+
No longer referenced by a   | type: int        |
                             | value: 3         |
                             +------------------+
```

The string did not replace or modify the integer object. Python only changed the binding of the name `a`. The old integer object's lifetime then depends on other references and CPython's internal caching.

> These diagrams are a conceptual model. CPython's actual memory layout is more complex.

In [ ]:
# A name can reference objects of different types

a = 3
print(a, type(a))

a = "hello"
print(a, type(a))

### Reference counting in CPython

CPython normally stores a reference count with each object. The count increases when another name or container starts referencing the object and decreases when a reference is removed.

When the count reaches zero, CPython can usually deallocate the object immediately. A separate cyclic garbage collector handles groups of unreachable objects that reference one another.

Some objects remain available because CPython caches, interns, or marks them as **immortal**. Small integers and some strings are common examples. This is runtime reuse—not delayed garbage collection.

```text
tracked --+
          +-------------> [one list object]
alias -----+

Two names now reference the same object.
After `del alias`, only `tracked` references it.
```

`sys.getrefcount(object)` temporarily creates one extra reference for its argument while the function runs. Its exact result is therefore an implementation detail and can change across Python versions; understand the change in the count rather than memorizing a number.

In [29]:
# Observe reference-count changes using a controlled object

import sys

tracked = []
print(sys.getrefcount(tracked))  # Includes the temporary function reference

alias = tracked
print(sys.getrefcount(tracked))  # Increases because alias points to it

del alias
print(sys.getrefcount(tracked))  # Decreases after removing alias

2
3
2


### Rebinding an immutable object

Integers are immutable: an existing integer object cannot be changed. Consider `a = 5`, `b = 3`, followed by `a = a + 2`. The variable `b` is included only to show that rebinding `a` does not affect other variables.

**Step 1: Initial references**

```text
+---+                  +------------------+
| a | ---------------> | int object: 5    |
+---+                  +------------------+

+---+                  +------------------+
| b | ---------------> | int object: 3    |
+---+                  +------------------+
```

**Step 2: Evaluate the right side, `a + 2`**

Python reads the value `5` from `a`, adds the literal value `2`, and produces a result representing `7`. The variable `b` is not involved. During evaluation, `a` still points to `5`.

```text
a --------------------> [int object: 5]
b --------------------> [int object: 3]
calculated result ----> [int object: 7]
```

**Step 3: Assign the result back to `a`**

```text
+---+                  +------------------+
| a | ---------------> | int object: 7    |
+---+                  +------------------+

+---+                  +------------------+
| b | ---------------> | int object: 3    |
+---+                  +------------------+

+------------------+
| int object: 5    |  <- a no longer points to this object
+------------------+
```

The object representing `5` was never changed into `7`; only `a` was disconnected from it and rebound to the result. In the general case, an object with no remaining references becomes eligible for deallocation. In CPython, small integers such as `5` and `7` are normally cached or immortal, so those particular objects may already exist and remain in memory.

In [ ]:
# Rebinding after an immutable operation

a = 5
b = 3
print(a, b)

a = a + 2
print(a, b)

### Shared reference followed by rebinding

This example has several separate steps. Rebinding a name changes its reference; mutating a list changes the list object itself.

**Step 1: Both names point to one list object**

```text
myListOne --+
             +-----------> +----------------------+
myListTwo --+              | List A: [1, 2, 3]   |
                            +----------------------+
```

**Step 2: `myListOne = "new"`**

Only `myListOne` is rebound. `myListTwo` still points to List A.

```text
myListOne ---------------> [str object: "new"]
myListTwo ---------------> [List A: [1, 2, 3]]
```

**Step 3: `myListOne = [1, 2, 3]`**

A separate list is assigned to `myListOne`. The two lists contain equal values but are different objects.

```text
myListOne ---------------> [List B: [1, 2, 3]]
myListTwo ---------------> [List A: [1, 2, 3]]
```

**Step 4: `myListOne[0] = 33`**

The mutation affects only List B.

```text
myListOne ---------------> [List B: [33, 2, 3]]
myListTwo ---------------> [List A: [ 1, 2, 3]]
```

In [ ]:
# Rebinding one name does not affect another name

myListOne = [1, 2, 3]
myListTwo = myListOne

myListOne = "new"
print(myListTwo)  # Still references the original list

myListOne = [1, 2, 3]  # Creates a separate list
myListOne[0] = 33

print(myListOne)
print(myListTwo)

### Mutation through a shared reference

The assignment `l2 = l1` copies the reference, not the list object. Both names therefore point to the same mutable list.

**Before `l1[0] = 99`**

```text
l1 --+
     +--------------------> [List A: [4, 5, 6]]
l2 --+
```

**After `l1[0] = 99`**

The list object itself is changed. Because both names still point to List A, the change is visible through both names.

```text
l1 --+
     +--------------------> [List A: [99, 5, 6]]
l2 --+
```

In [ ]:
# Both names reference the same mutable list

l1 = [4, 5, 6]
l2 = l1

l1[0] = 99
print(l1)
print(l2)

### Equal values in separate objects

**Step 1: `p2 = p1`**

Both names initially point to List A.

```text
p1 --+
     +--------------------> [List A: [7, 8, 9]]
p2 --+
```

**Step 2: `p2 = [7, 8, 9]`**

A new list expression creates List B, and `p2` is rebound to it. The two lists contain equal values but have separate identities.

```text
p1 ----------------------> [List A: [7, 8, 9]]
p2 ----------------------> [List B: [7, 8, 9]]
```

**Step 3: `p1[0] = 77`**

Only List A is mutated, so the value seen through `p2` remains unchanged.

```text
p1 ----------------------> [List A: [77, 8, 9]]
p2 ----------------------> [List B: [ 7, 8, 9]]
```

In [ ]:
# Equal-looking lists can be different objects

p1 = [7, 8, 9]
p2 = p1
p2 = [7, 8, 9]  # Rebind p2 to a new list

p1[0] = 77
print(p1)
print(p2)

### Shallow copies

The expression `h2 = h1[:]` creates a new outer list. It does not make `h2` another name for `h1`.

```text
h1 ----------------------> [List A: [1, 2, 3]]
h2 ----------------------> [List B: [1, 2, 3]]
```

After `h1[0] = 55`, only List A changes:

```text
h1 ----------------------> [List A: [55, 2, 3]]
h2 ----------------------> [List B: [ 1, 2, 3]]
```

Slicing and `copy.copy()` are **shallow copies**. They create a new outer container but reuse references to its elements. In this flat example, the elements are immutable integers, so that sharing is harmless. With nested mutable objects, the sharing becomes visible.

In [ ]:
# Copy a flat list using slicing

h1 = [1, 2, 3]
h2 = h1[:]

h1[0] = 55
print(h1)
print(h2)

### Shallow copy versus deep copy

For a nested list, `copy.copy()` creates a new outer list but keeps references to the same inner lists. `copy.deepcopy()` recursively creates independent copies in common cases.

```text
original --> [Outer A] --> [Inner X: [1, 2]]
                         --> [Inner Y: [3, 4]]

shallow  --> [Outer B] --> [same Inner X]
                         --> [same Inner Y]

deep     --> [Outer C] --> [Copied Inner X: [1, 2]]
                         --> [Copied Inner Y: [3, 4]]
```

Therefore, changing `original[0][0]` also appears in `shallow`, because both reach the same inner list. The change does not appear in `deep`, because its inner lists are independent copies.

In [ ]:
# Compare shallow and deep copies

import copy

original = [[1, 2], [3, 4]]
shallow = copy.copy(original)
deep = copy.deepcopy(original)

original[0][0] = 99
print(shallow)  # Inner list is shared
print(deep)     # Inner list is independent

### Equality versus identity

`==` asks: **Do these objects contain equal values?**

`is` asks: **Do these names point to the exact same object?**

**When `m = n`**

```text
n --+
    +---------------------> [List A: [1, 2, 3]]
m --+

n == m  -> True   (equal values)
n is m  -> True   (same object)
```

**After `m = [1, 2, 3]`**

```text
n -----------------------> [List A: [1, 2, 3]]
m -----------------------> [List B: [1, 2, 3]]

n == m  -> True    (equal values)
n is m  -> False   (different objects)
```

Use `is` for singleton checks such as `value is None`. Use `==` for normal value comparisons.

In [ ]:
# Compare value equality and object identity

n = [1, 2, 3]
m = n
print(n == m)  # True: same values
print(n is m)  # True: same object

m = [1, 2, 3]
print(n == m)  # True: same values
print(n is m)  # False: different objects